# Homework 23: Building PicoGPT

**Audience.** Students who understand causal multi-head attention and have previously built MLPs in PyTorch.

**Prerequisites.** Token and position embeddings; causal attention; PyTorch linear layers; ReLU; cross-entropy.

**Learning goals.** By the end, you will be able to:

- trace a pre-LayerNorm transformer block;
- assemble a stack of blocks into a decoder-only GPT;
- explain weight tying and next-token logits;
- verify causality through the complete model.

The implementation below is intentionally plain PyTorch: no Hugging Face model classes, FlashAttention, rotary embeddings, or key/value cache.


## Architecture

```text
token embedding + position embedding
             ↓
      LayerNorm → attention → + residual
             ↓
      LayerNorm → GELU MLP → + residual
             ↓       (repeat blocks)
      final LayerNorm
             ↓
      vocabulary logits
```

This is already a GPT architecture. “Pico” describes its classroom scale, not a different kind of network.


In [1]:
# S1: Imports and a small configuration object
from dataclasses import dataclass
import math

import torch
from torch import nn
from torch.nn import functional as F

_ = torch.manual_seed(158)

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 16
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    mlp_ratio: int = 4

    def __post_init__(self):
        assert self.d_model % self.n_heads == 0


## Three new building blocks

**LayerNorm** normalizes the channel coordinates of each token separately. Unlike BatchNorm, it does not combine examples or positions and does not maintain different running statistics for training and evaluation.

**GELU** is a smooth alternative to ReLU. Negative inputs are attenuated rather than always set exactly to zero.

A **residual connection** adds a sublayer's update back to its input. The shapes must match. This gives information and gradients a direct route through a deep stack.


In [2]:
# S1b: LayerNorm, GELU, and residual addition on one token tensor
ingredient_input = torch.tensor([
    [[1.0, 2.0, 3.0, 4.0], [4.0, 4.0, 6.0, 6.0]],
])
normalized = nn.LayerNorm(4)(ingredient_input)
activation_inputs = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print("mean of each normalized token:", normalized.mean(dim=-1))
print("std of each normalized token:", normalized.std(dim=-1, unbiased=False))
print("ReLU:", torch.relu(activation_inputs))
print("GELU:", nn.functional.gelu(activation_inputs).round(decimals=3))
print("residual shape:", tuple((ingredient_input + normalized).shape))


mean of each normalized token: tensor([[0., 0.]], grad_fn=<MeanBackward1>)
std of each normalized token: tensor([[1.0000, 1.0000]], grad_fn=<StdBackward0>)
ReLU: tensor([0., 0., 0., 1., 2.])
GELU: tensor([-0.0460, -0.1590,  0.0000,  0.8410,  1.9540])
residual shape: (1, 2, 4)


## 1. Attention and the MLP

Each block has two sublayers. Attention moves information between sequence positions. The MLP independently transforms the vector at each position. Both preserve `d_model`, allowing their outputs to be added back to the residual stream.


In [3]:
# S2: Causal multi-head self-attention
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.number_of_heads = config.n_heads
        self.head_size = config.d_model // config.n_heads
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.projection = nn.Linear(config.d_model, config.d_model)
        mask = torch.tril(torch.ones(config.block_size, config.block_size, dtype=torch.bool))
        self.register_buffer("mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.number_of_heads, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.number_of_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.number_of_heads, self.head_size).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.mask[:, :, :T, :T], float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        attended = weights @ v
        attended = attended.transpose(1, 2).contiguous().view(B, T, C)
        return self.projection(attended)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_size = config.mlp_ratio * config.d_model
        self.layers = nn.Sequential(
            nn.Linear(config.d_model, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, config.d_model),
        )

    def forward(self, x):
        return self.layers(x)


## 2. A pre-LayerNorm transformer block

“Pre-LayerNorm” means normalization happens before each sublayer. The residual additions give information and gradients a direct path through the stack.


In [4]:
# S3: One transformer block
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_attention = nn.LayerNorm(config.d_model)
        self.attention = CausalSelfAttention(config)
        self.ln_mlp = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x):
        x = x + self.attention(self.ln_attention(x))
        x = x + self.mlp(self.ln_mlp(x))
        return x

block_config = GPTConfig(vocab_size=20)
block = TransformerBlock(block_config)
block_input = torch.randn(2, 10, block_config.d_model)
block_output = block(block_input)

print("block input shape:", tuple(block_input.shape))
print("block output shape:", tuple(block_output.shape))


block input shape: (2, 10, 32)
block output shape: (2, 10, 32)


## 3. The complete decoder-only transformer

The language-model head maps each final vector to one logit per vocabulary token. We reuse the token-embedding weight matrix for this projection. This is called **weight tying**: the model uses the same learned token geometry when reading and predicting tokens.

`ModuleList` is a list that also registers every block with PyTorch. A plain Python list would run in the loop, but its parameters would be invisible to `model.parameters()` and therefore to the optimizer. `ModuleList` stores and registers modules, but it does not run them: the explicit `for block in self.blocks` loop in `forward` defines the computation. By contrast, `Sequential` both stores its children and passes the input through them automatically; `FeedForward` uses it because that subnetwork really is a straight chain.


In [5]:
# S4: PicoGPT
class PicoGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.block_size, config.d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layers)
        ])
        self.final_layer_norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        # Both modules now refer to the same Parameter object.
        self.lm_head.weight = self.token_embedding.weight

    @staticmethod
    def _initialize_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, indices, targets=None):
        B, T = indices.shape
        if T > self.config.block_size:
            raise ValueError("The sequence exceeds block_size.")

        positions = torch.arange(T, device=indices.device)
        x = self.token_embedding(indices) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_layer_norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                targets.reshape(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate_greedily(self, indices, number_of_tokens):
        self.eval()
        for _ in range(number_of_tokens):
            context = indices[:, -self.config.block_size:]
            logits, _ = self(context)
            next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
            indices = torch.cat([indices, next_id], dim=1)
        return indices


In [6]:
# S5: Shapes, loss, parameter count, and weight tying
config = GPTConfig(vocab_size=20, block_size=16, d_model=32, n_heads=4, n_layers=2)
model = PicoGPT(config)

input_ids = torch.randint(0, config.vocab_size, (3, 12))
target_ids = torch.randint(0, config.vocab_size, (3, 12))
logits, loss = model(input_ids, target_ids)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
weights_are_tied = model.lm_head.weight is model.token_embedding.weight

print("input shape:", tuple(input_ids.shape))
print("logit shape:", tuple(logits.shape))
print("loss:", round(float(loss.detach()), 4))
print("parameters:", parameter_count)
print("weights are tied:", weights_are_tied)


input shape: (3, 12)
logit shape: (3, 12, 20)
loss: 3.0223
parameters: 26624
weights are tied: True


The logits have an extra vocabulary dimension. The single loss averages `batch × sequence` separate next-token predictions.


In [7]:
# S6: Causality must hold through the whole GPT
first = torch.tensor([[1, 2, 3, 4, 5, 6]])
second = torch.tensor([[1, 2, 3, 12, 13, 14]])

first_logits, _ = model(first)
second_logits, _ = model(second)
complete_model_prefix_difference = (
    first_logits[:, :3] - second_logits[:, :3]
).abs().max()

print(
    "largest difference on shared prefix:",
    float(complete_model_prefix_difference.detach()),
)
assert torch.allclose(first_logits[:, :3], second_logits[:, :3], atol=1e-6)


largest difference on shared prefix: 0.0


## 4. Generation is repeated next-token prediction

Starting with a prompt, we keep the last `block_size` tokens, predict one new token, append it, and repeat. This untrained model produces nonsense; Homework 24 will train the same architecture.


In [8]:
# S7: Greedy generation with an untrained model
vocabulary = [
    "<bos>", "<eos>", ".", ",", "Ava", "found", "a", "blue", "kite", "dog",
    "the", "garden", "said", "good", "day", "Ben", "red", "ball", "happy", "!",
]
prompt = torch.tensor([[0, 4, 5]])
generated_ids = model.generate_greedily(prompt, number_of_tokens=8)
generated_tokens = [vocabulary[index] for index in generated_ids[0].tolist()]
print(generated_tokens)


['<bos>', 'Ava', 'found', 'found', '.', '.', '.', '.', '.', '.', '.']


## Notebook checkpoints

These checks emphasize architecture and tensor flow, not the arbitrary words chosen by an untrained network.


In [9]:
# S8: Deterministic checkpoint record
checkpoint_23 = {
    "block_output_shape": tuple(block_output.shape),
    "logit_shape": tuple(logits.shape),
    "number_of_scored_tokens": target_ids.numel(),
    "parameter_count": parameter_count,
    "blocks_are_module_list": isinstance(model.blocks, nn.ModuleList),
    "number_of_blocks": len(model.blocks),
    "weights_are_tied": weights_are_tied,
    "prefix_difference": float(complete_model_prefix_difference.detach()),
    "generated_length": generated_ids.shape[1],
}
checkpoint_23


{'block_output_shape': (2, 10, 32),
 'logit_shape': (3, 12, 20),
 'number_of_scored_tokens': 36,
 'parameter_count': 26624,
 'blocks_are_module_list': True,
 'number_of_blocks': 2,
 'weights_are_tied': True,
 'prefix_difference': 0.0,
 'generated_length': 11}

## Pitfall and extension

**Pitfall.** Removing positional embeddings does not cause a shape error, but it removes the model's direct representation of order. Shape checks alone cannot detect every conceptual bug.

**Optional extension.** Count parameters separately for embeddings, one transformer block, and the final normalization. Then estimate the size of a four-layer model with `d_model=256` and a 4,096-token vocabulary.

The course folder also contains `pico_gpt.py`, which collects the model, tokenizer, training helpers, activation cache, and intervention hook in one readable Python file.
